# 01 - Data exploration

Run the `prices` stage first. This notebook checks the canonical price panel, trading-day coverage and the next-day label balance.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np

prices = pd.read_parquet(ROOT / 'data/raw/prices.parquet')
prices['date'] = pd.to_datetime(prices['date'])
prices.shape, prices['ticker'].unique()

In [ ]:
prices.groupby('ticker').agg(
    start=('date', 'min'), end=('date', 'max'), rows=('date', 'size'),
    min_adj_close=('adj_close', 'min')
)

In [ ]:
# Missing trading days relative to the union calendar (no forward fill is applied).
calendar = prices['date'].drop_duplicates()
coverage = prices.groupby('ticker')['date'].nunique() / len(calendar)
coverage

In [ ]:
from src.labels import build_labels

labels = build_labels(prices)
print(labels['y_next_up'].mean())
labels.groupby('ticker')['y_next_up'].agg(['mean', 'count'])

In [ ]:
sorted_prices = prices.sort_values(['ticker', 'date'])
returns = np.log(sorted_prices['adj_close']).groupby(sorted_prices['ticker']).diff()
returns.describe()